# Lab 20 — Multi-Agent Research Demo Notebook

Notebook này giúp bạn **thử nghiệm nhanh** các khối logic của bài lab trước khi implement chính thức trong `src/`.

**Luồng làm việc:**
1. Khám phá schemas & shared state
2. Mock services (LLM + Search) để chạy không cần API key
3. Viết các agent demo (Researcher → Analyst → Writer)
4. Supervisor routing + vòng lặp workflow mini
5. Benchmark single-agent vs multi-agent


## 0. Setup

Chạy từ repo root với package đã cài (`pip install -e ".[dev]"`).

In [ ]:
import sys
from pathlib import Path

# Cho phép import package khi chạy notebook từ thư mục notebooks/
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "src"))

from multi_agent_research_lab.core.schemas import (
    AgentName,
    AgentResult,
    BenchmarkMetrics,
    ResearchQuery,
    SourceDocument,
)
from multi_agent_research_lab.core.state import ResearchState

print("✅ Import OK — package sẵn sàng")

## 1. Khám phá Shared State

`ResearchState` là **single source of truth** được truyền qua mọi agent. Mỗi agent đọc state, cập nhật, rồi trả lại.

In [1]:
query = ResearchQuery(
    query="So sánh RAG và fine-tuning cho domain adaptation",
    max_sources=3,
)
state = ResearchState(request=query)

state.record_route("researcher")
state.add_trace_event("demo", {"note": "first route recorded"})

print("Iteration:", state.iteration)
print("Route history:", state.route_history)
print("Trace:", state.trace)

NameError: name 'ResearchQuery' is not defined

## 2. Mock Services

Để demo không cần API key, ta dùng mock. Trong bản chính thức (`src/services/`), bạn sẽ nối provider thật (OpenAI / Tavily...).

- `MockSearchClient`: **đã viết sẵn** làm mẫu.
- `MockLLMClient`: Hoàn thiện giả lập phân nhánh phản hồi theo vai trò.

In [ ]:
from dataclasses import dataclass


class MockSearchClient:
    """Trả về nguồn giả lập — cùng interface với services.search_client.SearchClient."""

    _FAKE_DOCS = [
        SourceDocument(
            title="RAG vs Fine-tuning: A Practical Guide",
            url="https://example.com/rag-vs-ft",
            snippet="RAG phù hợp khi dữ liệu thay đổi thường xuyên; fine-tuning tốt cho style/format.",
        ),
        SourceDocument(
            title="Retrieval-Augmented Generation Survey",
            url="https://example.com/rag-survey",
            snippet="RAG giảm hallucination bằng cách grounding vào tài liệu ngoài.",
        ),
        SourceDocument(
            title="When to Fine-tune LLMs",
            url="https://example.com/when-finetune",
            snippet="Fine-tuning hiệu quả khi cần hành vi nhất quán và latency thấp.",
        ),
    ]

    def search(self, query: str, max_results: int = 5) -> list[SourceDocument]:
        return self._FAKE_DOCS[:max_results]


@dataclass(frozen=True)
class MockLLMResponse:
    content: str
    input_tokens: int | None = None
    output_tokens: int | None = None


class MockLLMClient:
    """Giả lập LLM — cùng interface với services.llm_client.LLMClient."""

    def complete(self, system_prompt: str, user_prompt: str) -> MockLLMResponse:
        sys_lower = system_prompt.lower()
        in_tokens = (len(system_prompt) + len(user_prompt)) // 4
        if "analyst" in sys_lower:
            content = (
                "1. RAG lý tưởng cho kho dữ liệu động và cần trích dẫn chính xác.\n"
                "2. Fine-tuning tối ưu cho phong cách trả lời cố định và độ trễ thấp.\n"
                "3. Kết hợp Hybrid RAG + Fine-tuning cho hiệu quả cao nhất."
            )
        elif "writer" in sys_lower:
            content = (
                "# Tổng hợp so sánh RAG và Fine-tuning\n\n"
                "RAG giúp tra cứu tài liệu linh hoạt và giảm ảo giác [1, 2]. "
                "Trong khi đó, Fine-tuning giúp đồng bộ phong cách và tối ưu hiệu năng [3].\n\n"
                "### Nguồn tham khảo:\n"
                "[1] RAG vs Fine-tuning: A Practical Guide (https://example.com/rag-vs-ft)\n"
                "[2] Retrieval-Augmented Generation Survey (https://example.com/rag-survey)\n"
                "[3] When to Fine-tune LLMs (https://example.com/when-finetune)"
            )
        else:
            content = "Phân tích tổng quan: RAG phù hợp dữ liệu thay đổi; Fine-tuning phù hợp cho phong cách và hành vi nhất quán."
        out_tokens = len(content) // 4
        return MockLLMResponse(content=content, input_tokens=in_tokens, output_tokens=out_tokens)


# Smoke test phần đã cho sẵn
search_client = MockSearchClient()
docs = search_client.search(query.query, max_results=query.max_sources)
for d in docs:
    print(f"- {d.title}: {d.snippet[:60]}...")

## 3. Demo Agents

Mỗi agent tuân theo contract `BaseAgent.run(state) -> state`.

- `DemoResearcherAgent`: gọi search, ghi `sources` + `research_notes`.
- `DemoAnalystAgent`: tổng hợp `sources` thành `analysis_notes`.
- `DemoWriterAgent`: viết `final_answer` kèm citation.

In [ ]:
class DemoResearcherAgent:
    """MẪU: thu thập nguồn và ghi chú nghiên cứu."""

    name = "researcher"

    def __init__(self, search_client: MockSearchClient) -> None:
        self.search_client = search_client

    def run(self, state: ResearchState) -> ResearchState:
        docs = self.search_client.search(
            state.request.query, max_results=state.request.max_sources
        )
        state.sources = docs
        state.research_notes = "\n".join(f"- {d.title}: {d.snippet}" for d in docs)
        state.agent_results.append(
            AgentResult(
                agent=AgentName.RESEARCHER,
                content=state.research_notes,
                metadata={"num_sources": len(docs)},
            )
        )
        state.add_trace_event("researcher.done", {"num_sources": len(docs)})
        return state


class DemoAnalystAgent:
    """Phân tích sources thành analysis_notes."""

    name = "analyst"

    def __init__(self, llm_client: MockLLMClient) -> None:
        self.llm_client = llm_client

    def run(self, state: ResearchState) -> ResearchState:
        if not state.sources:
            state.errors.append("DemoAnalystAgent: Không có sources")
            return state
        response = self.llm_client.complete(
            system_prompt="You are an analyst comparing research sources",
            user_prompt=state.research_notes or "",
        )
        state.analysis_notes = response.content
        state.agent_results.append(
            AgentResult(agent=AgentName.ANALYST, content=response.content)
        )
        state.add_trace_event("analyst.done", {"length": len(response.content)})
        return state


class DemoWriterAgent:
    """Viết final_answer có trích dẫn nguồn."""

    name = "writer"

    def __init__(self, llm_client: MockLLMClient) -> None:
        self.llm_client = llm_client

    def run(self, state: ResearchState) -> ResearchState:
        context = state.analysis_notes or state.research_notes or ""
        response = self.llm_client.complete(
            system_prompt="You are a writer synthesizing reports with citations",
            user_prompt=f"{state.request.query}\nContext:\n{context}",
        )
        state.final_answer = response.content
        state.agent_results.append(
            AgentResult(agent=AgentName.WRITER, content=response.content)
        )
        state.add_trace_event("writer.done", {"length": len(response.content)})
        return state


# Smoke test agent
state = ResearchState(request=query)
state = DemoResearcherAgent(search_client).run(state)
print(state.research_notes)

## 4. Supervisor Routing

Supervisor quyết định agent nào chạy tiếp dựa trên state hiện tại.

In [ ]:
MAX_ITERATIONS = 6


def demo_supervisor_route(state: ResearchState) -> str:
    """Trả về một trong: 'researcher' | 'analyst' | 'writer' | 'done'."""
    if state.iteration >= MAX_ITERATIONS:
        return "done"

    if not state.sources:
        return "researcher"
    if not state.analysis_notes:
        return "analyst"
    if not state.final_answer:
        return "writer"
    return "done"

## 5. Mini Workflow Loop

Vòng lặp điều phối mini workflow chạy end-to-end.

In [ ]:
def run_demo_workflow(query_text: str) -> ResearchState:
    q = ResearchQuery(query=query_text, max_sources=3)
    state = ResearchState(request=q)

    llm = MockLLMClient()
    agents = {
        "researcher": DemoResearcherAgent(MockSearchClient()),
        "analyst": DemoAnalystAgent(llm),
        "writer": DemoWriterAgent(llm),
    }

    while True:
        route = demo_supervisor_route(state)
        state.record_route(route)
        if route == "done":
            break
        state = agents[route].run(state)

    return state


final_state = run_demo_workflow("So sánh RAG và fine-tuning cho domain adaptation")
print("Route history:", final_state.route_history)
print("\n=== FINAL ANSWER ===\n")
print(final_state.final_answer)

## 6. Benchmark: Single-agent vs Multi-agent

Dùng `run_benchmark` từ package để so sánh.

In [ ]:
from multi_agent_research_lab.evaluation.benchmark import run_benchmark


def run_single_agent(query_text: str) -> ResearchState:
    """Baseline: một lần gọi LLM duy nhất, không search, không phân tích."""
    state = ResearchState(request=ResearchQuery(query=query_text))
    res = MockLLMClient().complete("You are a baseline agent", query_text)
    state.final_answer = res.content
    return state


def compute_citation_coverage(state: ResearchState) -> float:
    """Tỷ lệ nguồn trong state.sources được nhắc đến trong final_answer."""
    if not state.sources or not state.final_answer:
        return 0.0
    cited = sum(1 for d in state.sources if d.title.lower() in state.final_answer.lower() or f"[{state.sources.index(d)+1}]" in state.final_answer)
    return cited / len(state.sources)


demo_query = "So sánh RAG và fine-tuning cho domain adaptation"
results: list[BenchmarkMetrics] = []
for run_name, runner in [
    ("single_agent", run_single_agent),
    ("multi_agent", run_demo_workflow),
]:
    st, metrics = run_benchmark(run_name, demo_query, runner)
    metrics.citation_coverage = compute_citation_coverage(st)
    results.append(metrics)

print(f"{'run':<15}{'latency (s)':<15}{'citation cov.':<15}")
for m in results:
    print(f"{m.run_name:<15}{m.latency_seconds:<15.3f}{m.citation_coverage!s:<15}")

## 7. Next Steps — chuyển sang `src/`

Đã đồng bộ sang `src/multi_agent_research_lab/`!